In [1]:
# === ASSEMBLY CELL 1: The spine — crosswalk + verify element join holds ===
import pandas as pd
import numpy as np

BASE = r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot"

# 1. Load the crosswalk (the 2025-26 ID bridge: element <-> player_id <-> understat_id)
cw = pd.read_csv(BASE + r"\data\history\player_id_crosswalk_final.csv")
print("Crosswalk shape:", cw.shape)
print("Columns:", cw.columns.tolist())
print("\nUnderstat IDs present:", cw["understat_id"].notna().sum(), "of", len(cw),
      f"({cw['understat_id'].notna().mean():.1%}) — rest are bench players, expected")
print(cw.head())

# 2. Load 2025-26 vaastav rows and check the element join
df = pd.read_parquet(BASE + r"\data\history\all_seasons_fixed.parquet")
v2526 = df[(df["season"]=="2025-26") & (df["position"]!="AM")].copy()
vaastav_elements = set(pd.to_numeric(v2526["element"], errors="coerce").dropna().astype(int))
cw_elements = set(cw["element"].astype(int))

print(f"\n--- SPINE CHECK: does crosswalk element match 2025-26 vaastav element? ---")
print(f"Vaastav 2025-26 unique elements: {len(vaastav_elements)}")
print(f"Crosswalk elements:              {len(cw_elements)}")
print(f"Overlap (in both):               {len(vaastav_elements & cw_elements)}")
print(f"In crosswalk but not vaastav:    {len(cw_elements - vaastav_elements)}")
print(f"In vaastav but not crosswalk:    {len(vaastav_elements - cw_elements)}")

Crosswalk shape: (841, 4)
Columns: ['element', 'player_id', 'understat_id', 'matched_name']

Understat IDs present: 525 of 841 (62.4%) — rest are bench players, expected
   element  player_id  understat_id       matched_name
0      234        234           NaN    Aarón Anselmino
1      116        116        8942.0       Aaron Hickey
2      674        674        5603.0     Aaron Ramsdale
3      211        211           NaN       Aaron Ramsey
4      610        610        5584.0  Aaron Wan-Bissaka

--- SPINE CHECK: does crosswalk element match 2025-26 vaastav element? ---
Vaastav 2025-26 unique elements: 841
Crosswalk elements:              841
Overlap (in both):               841
In crosswalk but not vaastav:    0
In vaastav but not crosswalk:    0


In [2]:
# === ASSEMBLY CELL 2: Build the canonical player-gameweek skeleton (2025-26) ===
# One row per (player, gameweek). Carries all IDs + position + team.
# Every component's output attaches to this.

# Start from vaastav 2025-26 (has player-GW grain, element, position, team, minutes)
skel = v2526[["element", "GW", "name", "position", "team", "minutes", "total_points"]].copy()
skel["element"] = pd.to_numeric(skel["element"], errors="coerce").astype(int)
skel = skel.rename(columns={"GW": "gw", "total_points": "actual_points"})

# Attach the other IDs from the crosswalk (element -> player_id, understat_id)
skel = skel.merge(cw[["element", "player_id", "understat_id"]], on="element", how="left")

print("Skeleton shape:", skel.shape)
print("Grain check — rows per (element, gw) [should be 1, or 2 for double-GWs]:")
print(skel.groupby(["element","gw"]).size().value_counts())

print("\nID coverage:")
print(f"  player_id present:   {skel['player_id'].notna().mean():.1%}")
print(f"  understat_id present: {skel['understat_id'].notna().mean():.1%} (rest = bench, expected)")

print("\nPosition distribution:")
print(skel["position"].value_counts())

print("\nSample rows:")
print(skel.head(6).to_string(index=False))

Skeleton shape: (29757, 9)
Grain check — rows per (element, gw) [should be 1, or 2 for double-GWs]:
1    28919
2      419
Name: count, dtype: int64

ID coverage:
  player_id present:   100.0%
  understat_id present: 64.6% (rest = bench, expected)

Position distribution:
position
MID    13310
DEF     9733
GK      3427
FWD     3287
Name: count, dtype: int64

Sample rows:
 element  gw                           name position        team  minutes  actual_points  player_id  understat_id
     541   1               Reinildo Mandava      DEF  Sunderland       90              6        541        7432.0
      57   1                   Lewis Dobbin      MID Aston Villa        0              0         57           NaN
      87   1                  Ryan Christie      MID Bournemouth        0              0         87       10744.0
     216   1                   Zeki Amdouni      FWD     Burnley        0              0        216       11701.0
     612   1 Lucas Tolentino Coelho de Lima      MID    We

In [4]:
# %% [markdown]
# # Minutes Model — Assembly Prediction Pipeline (predicts 2025-26)
#
# Same pipeline as `minutes_predict.py`, but adapted for **assembly / live use**:
# trains on **all prior seasons (2022-23, 2023-24, 2024-25)** and predicts the live
# **2025-26** season. (The train-2/validate-1 split was for *measuring* the model —
# that's done, metrics are in the log. For prediction we use all history.)
#
# **2025-26 is the sealed test season** — we predict it for the live tool but must
# NOT tune configs against it.
#
# Ends with `mins_out`: `(element, gw, p_start, p60, e_minutes)` for 2025-26,
# ready to join onto the assembly skeleton.

# %%
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression

BASE = r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot"
TRAIN_SEASONS = ["2022-23", "2023-24", "2024-25"]   # all history for live prediction
PREDICT_SEASON = "2025-26"

# %% [markdown]
# ## 1. Load, filter, collapse double-GWs

# %%
df = pd.read_parquet(BASE + r"\data\history\all_seasons_fixed.parquet")
df = df[df["position"] != "AM"].copy()
model_df = df[df["starts"].notna()].copy()

key = ["season", "element", "GW"]
model_df = model_df.sort_values(key).reset_index(drop=True)
model_df["is_double_gw"] = (model_df.groupby(key)["minutes"].transform("size") > 1).astype(int)
agg_rules = {"minutes": "sum", "total_points": "sum", "starts": "max", "is_double_gw": "max"}
for col in model_df.columns:
    if col not in key and col not in agg_rules:
        agg_rules[col] = "first"
collapsed = model_df.groupby(key, as_index=False).agg(agg_rules)
assert collapsed.groupby(key).size().max() == 1
print(f"Rows after collapse: {len(collapsed)}")

# %% [markdown]
# ## 2. Features (shift-then-roll, no leakage)

# %%
collapsed = collapsed.sort_values(["season", "element", "GW"]).reset_index(drop=True)
collapsed["minutes_capped"] = collapsed["minutes"].clip(upper=90)
grp = collapsed.groupby(["season", "element"])

def rolling_prior(col, window, how):
    return grp[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).agg(how))

collapsed["started_last_gw"] = grp["starts"].shift(1)
collapsed["starts_last3"]    = rolling_prior("starts", 3, "sum")
collapsed["starts_last5"]    = rolling_prior("starts", 5, "sum")
collapsed["avg_min_last3"]   = rolling_prior("minutes_capped", 3, "mean")
collapsed["avg_min_last5"]   = rolling_prior("minutes_capped", 5, "mean")

def zero_run(s):
    prior = s.shift(1); is_zero = (prior == 0).astype(float)
    return is_zero.groupby((is_zero == 0).cumsum()).cumsum()
collapsed["consec_zero_mins"] = grp["minutes"].transform(zero_run)

def gws_since_start(s):
    prior = s.shift(1)
    return prior.groupby((prior == 1).cumsum()).cumcount()
collapsed["gws_since_last_start"] = grp["starts"].transform(gws_since_start)

collapsed["minutes_trend_3"] = grp["minutes_capped"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean() - s.shift(1).rolling(8, min_periods=2).mean())
collapsed["position_code"] = collapsed["position"].astype("category").cat.codes
print("Features built.")

# %% [markdown]
# ## 3. Cold-start + prior-season fallback, train P(start) on 2022-25

# %%
s1_feats = ["started_last_gw", "starts_last3", "starts_last5", "avg_min_last3",
            "avg_min_last5", "is_double_gw", "consec_zero_mins", "gws_since_last_start",
            "position_code", "value", "minutes_trend_3"]

season_agg = (collapsed.groupby(["season", "name"])
              .agg(prev_start_rate=("starts", "mean"),
                   prev_avg_minutes=("minutes_capped", "mean"),
                   prev_games=("starts", "size")).reset_index())
order = ["2022-23", "2023-24", "2024-25", "2025-26"]
prev_map = {order[i]: order[i-1] for i in range(1, len(order))}
season_agg["season"] = season_agg["season"].map({v: k for k, v in prev_map.items()})
season_agg = season_agg.dropna(subset=["season"])

cs = collapsed.copy()
cs["has_no_history"] = cs[["started_last_gw", "avg_min_last3"]].isna().any(axis=1).astype(int)
cs[s1_feats] = cs[s1_feats].fillna(0)
cs = cs.merge(season_agg, on=["season", "name"], how="left")
cs["transfer_status"] = np.where(cs["prev_start_rate"].isna(), 2, 0)
cs[["prev_start_rate", "prev_avg_minutes", "prev_games"]] = cs[["prev_start_rate", "prev_avg_minutes", "prev_games"]].fillna(0)

feats_pstart = s1_feats + ["has_no_history", "prev_start_rate", "prev_avg_minutes", "prev_games", "transfer_status"]
tr = cs[cs["season"].isin(TRAIN_SEASONS)]
model_pstart = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                  random_state=42, verbose=-1).fit(tr[feats_pstart], tr["starts"])
print(f"P(start) trained on {len(tr)} rows ({TRAIN_SEASONS})")

# %% [markdown]
# ## 4. P(60+|started) — trained on 2022-25

# %%
started_df = collapsed[collapsed["starts"] == 1].copy()
started_df = started_df.sort_values(["season", "element", "GW"]).reset_index(drop=True)
started_df["played_60"] = (started_df["minutes_capped"] >= 60).astype(int)
g2 = started_df.groupby(["season", "element"])
started_df["past60_rate_3"]      = g2["played_60"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
started_df["past60_rate_5"]      = g2["played_60"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
started_df["last_start_minutes"] = g2["minutes_capped"].shift(1)

s2_feats = s1_feats + ["past60_rate_3", "past60_rate_5", "last_start_minutes"]
d2 = started_df.dropna(subset=s2_feats)
tr2 = d2[d2["season"].isin(TRAIN_SEASONS)]
model_p60 = lgb.LGBMClassifier(n_estimators=100, num_leaves=7, min_child_samples=100,
                               reg_lambda=1.0, learning_rate=0.05, random_state=42, verbose=-1).fit(tr2[s2_feats], tr2["played_60"])
print(f"P(60+|started) trained on {len(tr2)} rows")

# %% [markdown]
# ## 5. P(came on|benched) + isotonic — trained on 2022-25

# %%
bench_df = collapsed[collapsed["starts"] == 0].copy()
bench_df["came_on"] = (bench_df["minutes"] > 0).astype(int)
sub_feats = ["avg_min_last3", "avg_min_last5", "minutes_trend_3", "consec_zero_mins",
             "gws_since_last_start", "starts_last3", "starts_last5", "started_last_gw",
             "position_code", "value", "is_double_gw"]
d3 = bench_df.dropna(subset=sub_feats)
tr3 = d3[d3["season"].isin(TRAIN_SEASONS)]
model_psub = lgb.LGBMClassifier(n_estimators=200, num_leaves=15, min_child_samples=50,
                                learning_rate=0.05, random_state=42, verbose=-1).fit(tr3[sub_feats], tr3["came_on"])
# calibrate on the training seasons' own predictions (held-out slice not needed for live use)
iso_sub = IsotonicRegression(out_of_bounds="clip").fit(
    model_psub.predict_proba(tr3[sub_feats])[:, 1], tr3["came_on"])
print(f"P(came on|benched) trained on {len(tr3)} rows")

# %% [markdown]
# ## 6. E[min|started] and E[min|sub] — trained on 2022-25

# %%
reg_feats = s1_feats
d4 = started_df.dropna(subset=reg_feats)
tr4 = d4[d4["season"].isin(TRAIN_SEASONS)]
model_min_start = lgb.LGBMRegressor(n_estimators=200, num_leaves=15, min_child_samples=50,
                                    learning_rate=0.05, random_state=42, verbose=-1).fit(tr4[reg_feats], tr4["minutes_capped"])

subs_df = bench_df[bench_df["came_on"] == 1].copy()
sub_reg_feats = ["avg_min_last3", "avg_min_last5", "minutes_trend_3", "consec_zero_mins",
                 "gws_since_last_start", "position_code", "value", "is_double_gw"]
d5 = subs_df.dropna(subset=sub_reg_feats)
tr5 = d5[d5["season"].isin(TRAIN_SEASONS)]
model_min_sub = lgb.LGBMRegressor(n_estimators=100, num_leaves=7, min_child_samples=100,
                                  reg_lambda=1.0, learning_rate=0.05, random_state=42, verbose=-1).fit(tr5[sub_reg_feats], tr5["minutes_capped"])
print("E[min|started] and E[min|sub] trained.")

# %% [markdown]
# ## 7. Predict 2025-26 → mins_out (ready for assembly)

# %%
starter_rate_cols = ["past60_rate_3", "past60_rate_5", "last_start_minutes"]
cs = cs.merge(started_df[["season", "element", "GW"] + starter_rate_cols],
              on=["season", "element", "GW"], how="left")
cs[starter_rate_cols] = cs[starter_rate_cols].fillna(0)

pf = cs[cs["season"] == PREDICT_SEASON].dropna(subset=sub_feats + reg_feats).copy()
pf["p_start"]   = model_pstart.predict_proba(pf[feats_pstart])[:, 1]
pf["p60"]       = model_p60.predict_proba(pf[s2_feats])[:, 1]
pf["p_sub"]     = iso_sub.predict(model_psub.predict_proba(pf[sub_feats])[:, 1])
pf["min_start"] = model_min_start.predict(pf[reg_feats])
pf["min_sub"]   = model_min_sub.predict(pf[sub_reg_feats])
pf["e_minutes"] = pf["p_start"] * pf["min_start"] + (1 - pf["p_start"]) * pf["p_sub"] * pf["min_sub"]

mins_out = pf[["element", "GW", "name", "position", "p_start", "p60", "e_minutes"]].rename(columns={"GW": "gw"})
print(f"\nMinutes predicted for {PREDICT_SEASON}: {len(mins_out)} player-GWs")
print(f"Mean e_minutes: {mins_out['e_minutes'].mean():.1f}")
print("\nTop e_minutes (should be nailed players):")
print(mins_out.sort_values("e_minutes", ascending=False).head(6).to_string(index=False))

Rows after collapse: 101465
Features built.
P(start) trained on 72127 rows (['2022-23', '2023-24', '2024-25'])
P(60+|started) trained on 19429 rows
P(came on|benched) trained on 47253 rows
E[min|started] and E[min|sub] trained.

Minutes predicted for 2025-26: 29338 player-GWs
Mean e_minutes: 25.3

Top e_minutes (should be nailed players):
 element  gw                  name position  p_start      p60  e_minutes
     224  33 Marc Cucurella Saseta      DEF 0.984881 0.971930  88.722229
     381   2         Mohamed Salah      MID 0.983185 0.981367  87.787686
       1  26     David Raya Martín       GK 0.967678 0.987739  87.660393
     736  33  Gianluigi Donnarumma       GK 0.967634 0.987739  87.518509
     736  36  Gianluigi Donnarumma       GK 0.967634 0.987739  87.518509
     287   2       Jordan Pickford       GK 0.977727 0.980445  87.248522


In [5]:
# === ASSEMBLY CELL 4: Join minutes onto the skeleton ===
# skeleton key = (element, gw); mins_out key = (element, gw). Direct join.

# Collapse skeleton double-GWs to one row per (element, gw) first, to match minutes grain.
# (Minutes already collapsed doubles; skeleton still has 419 double rows.)
skel_collapsed = (skel.sort_values(["element","gw"])
                  .groupby(["element","gw"], as_index=False)
                  .agg({"name":"first","position":"first","team":"first",
                        "minutes":"sum","actual_points":"sum",
                        "player_id":"first","understat_id":"first"}))
print("Skeleton after collapse:", len(skel_collapsed), "player-GWs")

# Join minutes
asm = skel_collapsed.merge(
    mins_out[["element","gw","p_start","p60","e_minutes"]],
    on=["element","gw"], how="left")

print("Assembled rows:", len(asm))
print(f"Rows with minutes prediction: {asm['e_minutes'].notna().sum()} "
      f"({asm['e_minutes'].notna().mean():.1%})")
print(f"Rows MISSING minutes: {asm['e_minutes'].isna().sum()} (likely GW1 cold-start / no-history)")

# Where are the missing ones?
missing = asm[asm["e_minutes"].isna()]
if len(missing):
    print("\nMissing-minutes rows by gameweek (top 5):")
    print(missing["gw"].value_counts().head())

print("\nSample assembled rows:")
print(asm[asm["e_minutes"].notna()].head(6)[
    ["element","gw","name","position","team","e_minutes","p_start","p60"]].to_string(index=False))

Skeleton after collapse: 29338 player-GWs
Assembled rows: 29338
Rows with minutes prediction: 29338 (100.0%)
Rows MISSING minutes: 0 (likely GW1 cold-start / no-history)

Sample assembled rows:
 element  gw              name position    team  e_minutes  p_start      p60
       1   1 David Raya Martín       GK Arsenal  57.036650 0.665134 0.883133
       1   2 David Raya Martín       GK Arsenal  79.936048 0.891455 0.980445
       1   3 David Raya Martín       GK Arsenal  82.790917 0.928381 0.980445
       1   4 David Raya Martín       GK Arsenal  83.257949 0.933245 0.982108
       1   5 David Raya Martín       GK Arsenal  83.231036 0.933245 0.982108
       1   6 David Raya Martín       GK Arsenal  83.676782 0.936717 0.982108


In [6]:
# === ASSEMBLY CELL 5: Attacking rates (npxG/90, xA/90) joined via understat_id ===
# Produce 2025-26 shrunk attacking rates, join onto asm by understat_id.
# Players without understat_id (bench) get the position-average fallback.

us = pd.read_parquet(BASE + r"\data\history\understat_season_aggregates.parquet")
for c in ["time","goals","xG","assists","xA","shots","npg","npxG"]:
    us[c] = pd.to_numeric(us[c])

MIN_TIME = 450
def k_for(stat, pos_label):
    return 2 if (stat=="npxG" and pos_label=="F") else 10

def shrunk_rate(season, pos_label, stat):
    pool = us[(us["understat_season"]==season)
              & (us["position"].str.contains(pos_label, na=False))
              & (us["time"]>=MIN_TIME)].copy()
    if len(pool) < 10: return None, None
    prior = pool[stat].sum()/pool["time"].sum()*90
    raw = pool[stat]/pool["time"]*90
    n90 = pool["time"]/90
    w = n90/(n90+k_for(stat,pos_label))
    pool["shrunk"] = w*raw + (1-w)*prior
    return pool[["id","shrunk"]], prior

# Build 2025-26 rate table (Understat season "2025"), per position, with priors captured
SEASON_US = "2025"
npxg_rows, xa_rows, priors = [], [], {}
for pos in ["F","M","D"]:
    npxg, npxg_prior = shrunk_rate(SEASON_US, pos, "npxG")
    xa, xa_prior = shrunk_rate(SEASON_US, pos, "xA")
    if npxg is not None:
        npxg_rows.append(npxg.rename(columns={"shrunk":"npxg90"}))
        xa_rows.append(xa.rename(columns={"shrunk":"xa90"}))
        priors[pos] = {"npxg": npxg_prior, "xa": xa_prior}

npxg_tbl = pd.concat(npxg_rows); xa_tbl = pd.concat(xa_rows)
rates = npxg_tbl.merge(xa_tbl, on="id", how="outer")
rates = rates.rename(columns={"id":"understat_id"})
print(f"Attacking rates built for {len(rates)} players (2025-26)")

# Join onto asm by understat_id
asm["understat_id_num"] = pd.to_numeric(asm["understat_id"], errors="coerce")
rates["understat_id"] = pd.to_numeric(rates["understat_id"], errors="coerce")
asm = asm.merge(rates, left_on="understat_id_num", right_on="understat_id", how="left", suffixes=("","_r"))

# Fallback: players without a rate (no understat_id, or <450 min) get position-average prior
pos_to_label = {"FWD":"F","MID":"M","DEF":"D","GK":"D"}  # GK -> defender prior (near-zero attacking)
def fallback(row, stat):
    lab = pos_to_label.get(row["position"], "M")
    return priors.get(lab, priors["M"])[stat]

need_npxg = asm["npxg90"].isna()
need_xa = asm["xa90"].isna()
asm.loc[need_npxg, "npxg90"] = asm[need_npxg].apply(lambda r: fallback(r,"npxg"), axis=1)
asm.loc[need_xa, "xa90"] = asm[need_xa].apply(lambda r: fallback(r,"xa"), axis=1)

print(f"Rows with a real rate: {(~need_npxg).sum()} | fallback: {need_npxg.sum()}")
print("\nSample — attacking rates attached:")
print(asm[~need_npxg].sort_values("npxg90",ascending=False).head(6)[
    ["name","position","team","npxg90","xa90","e_minutes"]].to_string(index=False))

Attacking rates built for 760 players (2025-26)
Rows with a real rate: 27687 | fallback: 15826

Sample — attacking rates attached:
          name position     team   npxg90     xa90  e_minutes
Erling Haaland      FWD Man City 0.751159 0.162541  55.609509
Erling Haaland      FWD Man City 0.751159 0.162541  79.192653
Erling Haaland      FWD Man City 0.751159 0.162541  79.192653
Erling Haaland      FWD Man City 0.751159 0.162541  83.035911
Erling Haaland      FWD Man City 0.751159 0.162541  83.035911
Erling Haaland      FWD Man City 0.751159 0.162541  84.684045


In [7]:
# Show DISTINCT players by attacking rate (not repeated gameweeks)
distinct = (asm[~asm["npxg90"].isna()]
            .drop_duplicates("element")
            .sort_values("npxg90", ascending=False))
print("Top npxG/90 players (distinct):")
print(distinct.head(10)[["name","position","team","npxg90","xa90"]].to_string(index=False))
print("\nTop xA/90 players (distinct) — should be creators/playmakers:")
print(distinct.sort_values("xa90",ascending=False).head(8)[["name","position","team","npxg90","xa90"]].to_string(index=False))

Top npxG/90 players (distinct):
                            name position        team   npxg90     xa90
                  Erling Haaland      FWD    Man City 0.751159 0.162541
                     Kai Havertz      FWD     Arsenal 0.650852 0.179442
                  Benjamin Sesko      FWD     Man Utd 0.650179 0.074369
                   Donyell Malen      MID Aston Villa 0.617766 0.112033
Norberto Bercique Gomes Betuncal      FWD     Everton 0.600834 0.091836
                   Ollie Watkins      FWD Aston Villa 0.566661 0.101955
                    Hugo Ekitiké      FWD   Liverpool 0.560422 0.131269
                  Joshua Zirkzee      FWD     Man Utd 0.533268 0.165829
                   Callum Wilson      FWD    West Ham 0.524906 0.106723
                   William Osula      FWD   Newcastle 0.520103 0.104355

Top xA/90 players (distinct) — should be creators/playmakers:
                  name position           team   npxg90     xa90
Bruno Borges Fernandes      MID        Man Utd 0

In [8]:
print("Columns in asm:", asm.columns.tolist())
print("\nRow count:", len(asm))
print("\nSample with everything so far:")
print(asm.head(3)[["name","position","team","gw","e_minutes","p_start","p60","npxg90","xa90"]].to_string(index=False))

Columns in asm: ['element', 'gw', 'name', 'position', 'team', 'minutes', 'actual_points', 'player_id', 'understat_id', 'p_start', 'p60', 'e_minutes', 'understat_id_num', 'understat_id_r', 'npxg90', 'xa90']

Row count: 43513

Sample with everything so far:
             name position    team  gw  e_minutes  p_start      p60   npxg90     xa90
David Raya Martín       GK Arsenal   1  57.036650 0.665134 0.883133 0.068329 0.071336
David Raya Martín       GK Arsenal   2  79.936048 0.891455 0.980445 0.068329 0.071336
David Raya Martín       GK Arsenal   3  82.790917 0.928381 0.980445 0.068329 0.071336


In [9]:
# === FIX: dedupe the attacking join (asm grew from ~29k to 43k = duplicates) ===
print("Current asm rows:", len(asm), "(should be ~29,338)")

# Check: does the rates table have duplicate understat_ids?
print("Duplicate understat_ids in rates:", rates["understat_id"].duplicated().sum())

# Rebuild asm cleanly: dedupe rates first (keep highest npxg90 per understat_id — the primary position)
rates_clean = (rates.sort_values("npxg90", ascending=False)
               .drop_duplicates("understat_id", keep="first"))
print("Rates after dedupe:", len(rates_clean))

# Rebuild asm from the minutes-joined version (before the attacking join doubled it)
asm = skel_collapsed.merge(
    mins_out[["element","gw","p_start","p60","e_minutes"]], on=["element","gw"], how="left")
asm["understat_id_num"] = pd.to_numeric(asm["understat_id"], errors="coerce")
rates_clean["understat_id"] = pd.to_numeric(rates_clean["understat_id"], errors="coerce")
asm = asm.merge(rates_clean, left_on="understat_id_num", right_on="understat_id", how="left", suffixes=("","_r"))

# Re-apply fallback
need_npxg = asm["npxg90"].isna(); need_xa = asm["xa90"].isna()
asm.loc[need_npxg,"npxg90"] = asm[need_npxg].apply(lambda r: fallback(r,"npxg"), axis=1)
asm.loc[need_xa,"xa90"] = asm[need_xa].apply(lambda r: fallback(r,"xa"), axis=1)

print("\nasm rows after clean rebuild:", len(asm), "(should be ~29,338)")
print("Duplicate (element,gw) pairs:", asm.duplicated(["element","gw"]).sum(), "(should be 0)")

Current asm rows: 43513 (should be ~29,338)
Duplicate understat_ids in rates: 390
Rates after dedupe: 370

asm rows after clean rebuild: 29338 (should be ~29,338)
Duplicate (element,gw) pairs: 0 (should be 0)


In [10]:
# === Check 2025-26 fixture representation for the Dixon-Coles join ===
odds = pd.read_parquet(BASE + r"\data\history\odds_all_seasons.parquet")
o2526 = odds[odds["season"]=="2025-26"].copy()
print("2025-26 matches in odds file:", len(o2526))
print("Columns of interest:", [c for c in o2526.columns if c in ["Date","HomeTeam","AwayTeam","season","B365H"]])

# Is there a gameweek column? Or do we derive GW from date?
print("\nGameweek column present?", any("gw" in c.lower() or "week" in c.lower() or "round" in c.lower() for c in o2526.columns))
print("\nTeam names in odds (football-data.co.uk style):")
print(sorted(o2526["HomeTeam"].unique())[:10])

print("\nTeam names in our skeleton (vaastav style):")
print(sorted(asm["team"].unique())[:10])

2025-26 matches in odds file: 380
Columns of interest: ['Date', 'HomeTeam', 'AwayTeam', 'B365H', 'season']

Gameweek column present? False

Team names in odds (football-data.co.uk style):
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham']

Team names in our skeleton (vaastav style):
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham']


In [11]:
# === Find team-name mismatches + build the date->GW bridge ===

# 1. Which team names DON'T match between the two conventions?
odds_teams = set(o2526["HomeTeam"].unique())
skel_teams = set(asm["team"].unique())
print("In odds but NOT skeleton:", sorted(odds_teams - skel_teams))
print("In skeleton but NOT odds:", sorted(skel_teams - odds_teams))
print("Matching teams:", len(odds_teams & skel_teams), "of 20")

# 2. Build date -> GW map from the skeleton side.
# vaastav rows have kickoff dates? check what date info we have per player-GW
v = df[df["season"]=="2025-26"]
date_cols = [c for c in v.columns if "date" in c.lower() or "kickoff" in c.lower() or "time" in c.lower()]
print("\nDate columns in vaastav 2025-26:", date_cols)

In odds but NOT skeleton: ['Man United', 'Tottenham']
In skeleton but NOT odds: ['Man Utd', 'Spurs']
Matching teams: 18 of 20

Date columns in vaastav 2025-26: ['kickoff_time', 'kickoff_time_formatted']


In [14]:
# %% [markdown]
# # Dixon-Coles — Clean Prediction Pipeline (team goals, clean sheets, fixture scaling)
#
# Final config only. The investigation (no-decay vs decay gridsearch, DC low-score
# correction test, per-team home advantage, fixture-pairing tests) is stripped —
# see the earlier session notes for why each lost. This produces what the points
# model needs: **P(clean sheet)** per team per fixture, and **team goal
# expectations (λ)** used to scale each player's attacking rate by opponent.
#
# **Final config (from the build decisions):**
# - Hand-fit Poisson MLE: λ_home = exp(atk[home] + def[away] + home_adv),
#   λ_away = exp(atk[away] + def[home]); 2·n_teams + 2 params, scipy L-BFGS-B
# - **Time-decay: 1-year half-life** (won the gridsearch: val LL ≈ −3.01 vs −3.06 no-decay)
# - **DC low-score correction (ρ)**: kept for completeness, effect negligible (ρ≈−0.03)
# - **Odds blend**: market dominates DC on match outcomes (WDL ≈ 53.7% vs 48.9%);
#   for **clean sheets**, blend at **w≈0.2** (0.2·DC + 0.8·market) — best CS Brier ≈ 0.172
#
# **Output key:** fixture-level `(season, home, away)` → λ_home, λ_away, P(home CS),
# P(away CS). Team-name based; assembly maps team names to player rows.

# %%
import pandas as pd
import numpy as np
from scipy.stats import poisson
from scipy.optimize import minimize

BASE = r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot"
VAL_SEASON = "2025-26"
HALF_LIFE_DAYS = 365          # 1-year decay (won the gridsearch)
# Two different blend weights — the two outputs prefer different mixes (both verified):
#   - goal expectations (λ, used to scale attacking rates): pure market wins WDL
#     (53.7% at w=0, monotonically worse as DC is added) → LAM_BLEND_W = 0.0
#   - clean sheets: a small DC contribution helps (Brier 0.1718 at w=0.2) → CS_BLEND_W = 0.2
LAM_BLEND_W = 0.0            # fixture goal expectations: pure market
CS_BLEND_W = 0.2            # clean-sheet probabilities: 0.2*DC + 0.8*market

# %% [markdown]
# ## 1. Load odds/results data (has scorelines + Bet365 odds per match)

# %%
odds = pd.read_parquet(BASE + r"\data\history\odds_all_seasons.parquet")
matches = odds[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "season",
                "B365H", "B365D", "B365A"]].copy()
matches.columns = ["date", "home", "away", "home_goals", "away_goals", "season",
                   "b365h", "b365d", "b365a"]
matches["date_parsed"] = pd.to_datetime(matches["date"], dayfirst=True)
teams = sorted(set(matches["home"]) | set(matches["away"]))
print(f"{len(matches)} matches, {len(teams)} teams, seasons {sorted(matches['season'].unique())}")

# %% [markdown]
# ## 2. Fit Dixon-Coles with 1-year time-decay and low-score correction
# λ from attack/defence strengths + home advantage; recency-weighted likelihood;
# τ is the DC correction for low-scoring scorelines (0-0, 1-0, 0-1, 1-1).

# %%
def fit_dc_decay(train_matches, all_teams, ref_date, half_life_days):
    idx = {t: i for i, t in enumerate(all_teams)}
    nt = len(all_teams)
    h  = train_matches["home"].map(idx).values
    a  = train_matches["away"].map(idx).values
    hg = train_matches["home_goals"].values
    ag = train_matches["away_goals"].values
    age = (ref_date - train_matches["date_parsed"]).dt.days.values
    w = np.ones(len(age)) if half_life_days is None else np.exp(-(np.log(2)/half_life_days) * age)

    def nll(params):
        atk, dfc = params[:nt], params[nt:2*nt]
        hadv, rho = params[-2], params[-1]
        lam_h = np.exp(atk[h] + dfc[a] + hadv)
        lam_a = np.exp(atk[a] + dfc[h])
        log_p = poisson.logpmf(hg, lam_h) + poisson.logpmf(ag, lam_a)
        tau = np.ones(len(hg))
        tau[(hg==0)&(ag==0)] = (1 - lam_h*lam_a*rho)[(hg==0)&(ag==0)]
        tau[(hg==0)&(ag==1)] = (1 + lam_h*rho)[(hg==0)&(ag==1)]
        tau[(hg==1)&(ag==0)] = (1 + lam_a*rho)[(hg==1)&(ag==0)]
        tau[(hg==1)&(ag==1)] = (1 - rho)
        log_p = log_p + np.log(np.clip(tau, 1e-10, None))
        return -(w * log_p).sum()

    x0 = np.zeros(2*nt + 2); x0[-2] = 0.25
    res = minimize(nll, x0, method="L-BFGS-B")
    return res.x, idx, nt

# Train on everything before the validation season; ref date = val season start
train_m = matches[matches["season"] < VAL_SEASON].copy()
ref = matches[matches["season"] == VAL_SEASON]["date_parsed"].min()
params, idx, nt = fit_dc_decay(train_m, teams, ref, HALF_LIFE_DAYS)
atk, dfc, hadv, rho = params[:nt], params[nt:2*nt], params[-2], params[-1]
print(f"Fitted. home_adv={hadv:.3f}, rho={rho:.4f} (DC correction, expected ~ -0.03)")

# %% [markdown]
# ## 3. Invert market odds -> implied goal expectations (strip vig, solve 2-unknown system)

# %%
def outcomes_from_lambdas(lam_h, lam_a, max_goals=10):
    hp = poisson.pmf(np.arange(max_goals+1), lam_h)
    ap = poisson.pmf(np.arange(max_goals+1), lam_a)
    M = np.outer(hp, ap)
    return np.tril(M, -1).sum(), np.trace(M), np.triu(M, 1).sum()

def implied_lambdas(pH, pD, pA):
    def mismatch(log_lams):
        lam_h, lam_a = np.exp(log_lams)
        mH, mD, mA = outcomes_from_lambdas(lam_h, lam_a)
        return (mH-pH)**2 + (mD-pD)**2 + (mA-pA)**2
    res = minimize(mismatch, x0=[np.log(1.4), np.log(1.1)], method="Nelder-Mead")
    return np.exp(res.x)

val = matches[matches["season"] == VAL_SEASON].copy()
# de-vig the odds into probabilities
inv = 1/val[["b365h","b365d","b365a"]].values
val[["p_H","p_D","p_A"]] = inv / inv.sum(axis=1, keepdims=True)
# only fixtures whose teams the model knows
known = val["home"].isin(idx) & val["away"].isin(idx)
val = val[known].copy()
lam_pairs = np.array([implied_lambdas(r.p_H, r.p_D, r.p_A) for r in val.itertuples()])
val["mkt_lam_h"], val["mkt_lam_a"] = lam_pairs[:,0], lam_pairs[:,1]

# DC lambdas
hi = val["home"].map(idx).values; ai = val["away"].map(idx).values
val["dc_lam_h"] = np.exp(atk[hi] + dfc[ai] + hadv)
val["dc_lam_a"] = np.exp(atk[ai] + dfc[hi])

# %% [markdown]
# ## 4. The blended fixture model + clean-sheet validation
# Blend: λ = w·DC + (1-w)·market. P(clean sheet) = exp(-opponent λ).

# %%
def blend(df, w):
    """Blend DC and market lambdas: λ = w·DC + (1-w)·market."""
    lam_h = w * df["dc_lam_h"] + (1-w) * df["mkt_lam_h"]
    lam_a = w * df["dc_lam_a"] + (1-w) * df["mkt_lam_a"]
    return lam_h, lam_a

val["home_cs_actual"] = (val["away_goals"] == 0).astype(int)
val["away_cs_actual"] = (val["home_goals"] == 0).astype(int)

print("Clean-sheet Brier by blend weight (lower better):")
for w in [0.0, 0.2, 0.5, 1.0]:
    lh, la = blend(val, w)
    p_home_cs, p_away_cs = np.exp(-la), np.exp(-lh)
    brier = (((p_home_cs - val["home_cs_actual"])**2).mean() +
             ((p_away_cs - val["away_cs_actual"])**2).mean())/2
    tag = "pure market" if w==0 else ("pure DC" if w==1 else f"blend w={w}")
    print(f"  w={w:.1f}  Brier {brier:.4f}   {tag}")
base = val[["home_cs_actual","away_cs_actual"]].values.mean()
print(f"  baseline (flat {base:.2f}): Brier {((base-val[['home_cs_actual','away_cs_actual']].values)**2).mean():.4f}")
print("\nNote: clean sheets use w=0.2; goal expectations (λ) use w=0.0 (pure market — best WDL).")

# %% [markdown]
# ## 5. The reusable predict function (fixture -> λ and clean-sheet probs)

# %%
def predict_fixtures(season_matches):
    """Given fixtures with team names + market odds, return λ_home/away and
    P(home CS)/P(away CS) using the blended model. Key: (season, home, away)."""
    df = season_matches.copy()
    known = df["home"].isin(idx) & df["away"].isin(idx)
    df = df[known].copy()
    # market implied
    inv = 1/df[["b365h","b365d","b365a"]].values
    df[["p_H","p_D","p_A"]] = inv / inv.sum(axis=1, keepdims=True)
    lp = np.array([implied_lambdas(r.p_H, r.p_D, r.p_A) for r in df.itertuples()])
    df["mkt_lam_h"], df["mkt_lam_a"] = lp[:,0], lp[:,1]
    hi = df["home"].map(idx).values; ai = df["away"].map(idx).values
    df["dc_lam_h"] = np.exp(atk[hi] + dfc[ai] + hadv)
    df["dc_lam_a"] = np.exp(atk[ai] + dfc[hi])
    # goal expectations (for scaling attacking rates): pure market (best WDL)
    df["lam_home"], df["lam_away"] = blend(df, LAM_BLEND_W)
    # clean-sheet probabilities: small DC contribution helps → use CS_BLEND_W
    cs_lh, cs_la = blend(df, CS_BLEND_W)
    df["p_home_cs"] = np.exp(-cs_la)   # home CS = away scores 0
    df["p_away_cs"] = np.exp(-cs_lh)
    return df[["season","home","away","lam_home","lam_away","p_home_cs","p_away_cs"]]

out = predict_fixtures(val)
print(f"\nFixture predictions: {len(out)} matches")
print("Strongest clean-sheet fixtures (home):")
print(out.sort_values("p_home_cs", ascending=False)
      [["home","away","lam_home","lam_away","p_home_cs"]].head(6).to_string(index=False))

C:\Users\veers\AppData\Local\Temp\ipykernel_10556\1768802669.py:46: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  matches["date_parsed"] = pd.to_datetime(matches["date"], dayfirst=True)


3800 matches, 34 teams, seasons ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Fitted. home_adv=0.135, rho=-0.0041 (DC correction, expected ~ -0.03)
Clean-sheet Brier by blend weight (lower better):
  w=0.0  Brier 0.1809   pure market
  w=0.2  Brier 0.1799   blend w=0.2
  w=0.5  Brier 0.1798   blend w=0.5
  w=1.0  Brier 0.1825   pure DC
  baseline (flat 0.26): Brier 0.1901

Note: clean sheets use w=0.2; goal expectations (λ) use w=0.0 (pure market — best WDL).

Fixture predictions: 380 matches
Strongest clean-sheet fixtures (home):
    home       away  lam_home  lam_away  p_home_cs
 Arsenal   West Ham  2.345642  0.454749   0.593145
 Arsenal     Wolves  2.786285  0.515059   0.569509
 Chelsea Sunderland  1.958242  0.604477   0.552975
 Arsenal Sunderland  2.452920  0.647615   0.548895
 Arsenal    Everton  1.970892  0.642837   0.526043
Man City    Burnley  2.738416  0.648066   0.522704


In [15]:
# === ASSEMBLY CELL 6a: Build fixture table + team-name bridge ===

# 1. Team-name map: odds/DC style -> skeleton style
team_map = {"Man United": "Man Utd", "Tottenham": "Spurs"}
def norm_team(t): return team_map.get(t, t)

# 2. Run Dixon-Coles to get 2025-26 fixture predictions (λ + CS), normalize team names
#    (predict_fixtures from dixon_coles_predict.py — assumes it's been run; else inline)
o2526 = odds[odds["season"]=="2025-26"].copy()
o2526_pred = predict_fixtures(o2526.rename(columns={
    "HomeTeam":"home","AwayTeam":"away","FTHG":"home_goals","FTAG":"away_goals",
    "B365H":"b365h","B365D":"b365d","B365A":"b365a"}).assign(season="2025-26"))
o2526_pred["home"] = o2526_pred["home"].map(norm_team)
o2526_pred["away"] = o2526_pred["away"].map(norm_team)
o2526_pred["match_date"] = pd.to_datetime(o2526.loc[o2526_pred.index, "Date"], format="mixed", dayfirst=True).dt.date
print("DC fixture predictions:", len(o2526_pred))

# 3. Build (team, gw) -> date from vaastav kickoff_time
v = df[df["season"]=="2025-26"].copy()
v["match_date"] = pd.to_datetime(v["kickoff_time"]).dt.date
team_gw_date = v[["team","GW","match_date"]].drop_duplicates().rename(columns={"GW":"gw"})
print("Team-GW-date rows:", len(team_gw_date))
print(team_gw_date.head())

DC fixture predictions: 380
Team-GW-date rows: 760
               team  gw  match_date
224143   Sunderland   1  2025-08-16
224144  Aston Villa   1  2025-08-16
224145  Bournemouth   1  2025-08-15
224146      Burnley   1  2025-08-16
224147     West Ham   1  2025-08-16


In [16]:
# === ASSEMBLY CELL 6b: Join DC λ + P(CS) onto each team-gameweek ===

# Reshape DC predictions into team-level rows: each fixture -> 2 rows (home, away)
home_rows = o2526_pred[["home","match_date","lam_home","lam_away","p_home_cs"]].copy()
home_rows.columns = ["team","match_date","team_lambda","opp_lambda","p_cs"]
away_rows = o2526_pred[["away","match_date","lam_away","lam_home","p_away_cs"]].copy()
away_rows.columns = ["team","match_date","team_lambda","opp_lambda","p_cs"]
team_fixtures = pd.concat([home_rows, away_rows], ignore_index=True)
print("Team-fixture rows (should be ~760 = 380×2):", len(team_fixtures))

# Attach GW via the (team, match_date) -> gw bridge from vaastav
team_gw_date["match_date"] = pd.to_datetime(team_gw_date["match_date"]).dt.date
team_fixtures["match_date"] = pd.to_datetime(team_fixtures["match_date"]).dt.date
team_fixtures = team_fixtures.merge(team_gw_date, on=["team","match_date"], how="left")
print("Rows that got a GW:", team_fixtures["gw"].notna().sum(), "of", len(team_fixtures))
print("Unmatched (team-name or date issues):", team_fixtures["gw"].isna().sum())

# Now join onto asm by (team, gw)
asm = asm.merge(team_fixtures[["team","gw","team_lambda","opp_lambda","p_cs"]].drop_duplicates(["team","gw"]),
                on=["team","gw"], how="left")
print("\nasm rows:", len(asm), "| with fixture data:", asm["p_cs"].notna().sum())
print("\nSample — top clean-sheet players:")
print(asm[asm["p_cs"].notna()].sort_values("p_cs",ascending=False).drop_duplicates("element").head(6)[
    ["name","position","team","gw","team_lambda","p_cs"]].to_string(index=False))

Team-fixture rows (should be ~760 = 380×2): 760
Rows that got a GW: 760 of 760
Unmatched (team-name or date issues): 0

asm rows: 29338 | with fixture data: 29338

Sample — top clean-sheet players:
                   name position    team  gw  team_lambda     p_cs
      David Raya Martín       GK Arsenal   7     2.345642 0.593145
              Karl Hein       GK Arsenal   7     2.345642 0.593145
           Ismeal Kabia      MID Arsenal   7     2.345642 0.593145
     Cristhian Mosquera      DEF Arsenal   7     2.345642 0.593145
Martín Zubimendi Ibáñez      MID Arsenal   7     2.345642 0.593145
        Maldini Kacurri      DEF Arsenal   7     2.345642 0.593145


In [17]:
# === ASSEMBLY: Defensive Contribution predictions for 2025-26 ===
SEASON_DC = "2025-2026"   # Core-Insights uses this format
FWD_BASE_RATE = 0.005

ms = pd.read_parquet(BASE + r"\data\history\core_insights_matchstats.parquet")
ms_pl = ms[ms["match_id"].str.contains("-prem-", na=False)].copy()
for c in ["tackles","interceptions","recoveries","blocks","clearances","minutes_played"]:
    ms_pl[c] = pd.to_numeric(ms_pl[c], errors="coerce")

gwref = pd.read_parquet(BASE + r"\data\history\core_insights_gameweek_stats.parquet")
pos_map = gwref[["id","position"]].drop_duplicates("id").set_index("id")["position"]
ms_pl["position"] = ms_pl["player_id"].map(pos_map)

ms_pl["cbit"]  = ms_pl["clearances"] + ms_pl["blocks"] + ms_pl["interceptions"] + ms_pl["tackles"]
ms_pl["cbirt"] = ms_pl["cbit"] + ms_pl["recoveries"]
ms_pl["dc_metric"] = np.where(ms_pl["position"]=="Defender", ms_pl["cbit"], ms_pl["cbirt"])
ms_pl["dc_threshold"] = np.where(ms_pl["position"]=="Defender", 10, 12)
ms_pl["dc_hit"] = (ms_pl["dc_metric"] >= ms_pl["dc_threshold"]).astype(int)
played = ms_pl[ms_pl["minutes_played"] >= 1].copy()

dc_d = played[played["season"]==SEASON_DC].sort_values(["player_id","gw"]).reset_index(drop=True)
dc_d["dc_per90"] = dc_d["dc_metric"] / dc_d["minutes_played"].clip(lower=1) * 90
gdc = dc_d.groupby("player_id")
rpf = lambda c,w,h: gdc[c].transform(lambda s: s.shift(1).rolling(w,min_periods=1).agg(h))
dc_d["roll_dc90_3"]=rpf("dc_per90",3,"mean"); dc_d["roll_dc90_5"]=rpf("dc_per90",5,"mean")
dc_d["roll_hit_5"]=rpf("dc_hit",5,"mean"); dc_d["roll_mins_3"]=rpf("minutes_played",3,"mean")
dc_d["roll_dc90_3c"]=dc_d["roll_dc90_3"].clip(upper=30); dc_d["roll_dc90_5c"]=dc_d["roll_dc90_5"].clip(upper=30)
DC_FEATURES=["roll_dc90_3c","roll_dc90_5c","roll_hit_5","roll_mins_3"]

def _mk():
    return lgb.LGBMClassifier(n_estimators=150,num_leaves=15,min_child_samples=40,
                              learning_rate=0.05,random_state=42,verbose=-1)

# Walk-forward: predict every gameweek from prior gameweeks
dc_preds = []
for g in sorted(dc_d["gw"].unique()):
    for pos in ["Defender","Midfielder","Forward"]:
        te = dc_d[(dc_d["gw"]==g)&(dc_d["position"]==pos)].dropna(subset=DC_FEATURES).copy()
        if len(te)==0: continue
        if pos=="Forward":
            te["p_dc_hit"]=FWD_BASE_RATE
        else:
            prior=dc_d[(dc_d["gw"]<g)&(dc_d["position"]==pos)].dropna(subset=DC_FEATURES)
            if len(prior)<150:
                te["p_dc_hit"]=prior["dc_hit"].mean() if len(prior) else 0.13
            else:
                m=_mk().fit(prior[DC_FEATURES],prior["dc_hit"])
                te["p_dc_hit"]=m.predict_proba(te[DC_FEATURES])[:,1]
        dc_preds.append(te[["player_id","gw","p_dc_hit"]])

dc_out = pd.concat(dc_preds)
print("DC predictions for 2025-26:", len(dc_out))
print(dc_out.sort_values("p_dc_hit",ascending=False).head(5).to_string(index=False))

DC predictions for 2025-26: 9979
 player_id  gw  p_dc_hit
       490  10  0.798085
       717  11  0.778342
       408   9  0.770717
       387   9  0.763716
       106  22  0.751665


In [18]:
# === ASSEMBLY CELL 7: Join defensive contribution onto asm ===
# DC keyed on (player_id, gw); asm's player_id == element. Direct join.
asm = asm.merge(dc_out, on=["player_id","gw"], how="left")

print("asm rows:", len(asm))
print(f"Rows with a DC prediction: {asm['p_dc_hit'].notna().sum()} ({asm['p_dc_hit'].notna().mean():.1%})")
print(f"Rows missing DC (need fallback): {asm['p_dc_hit'].isna().sum()}")

# Fallback for players without a DC prediction (bench / no history / GK):
# position base rates from the log — DEF 12.5%, MID 13.6%, FWD 5.8%, GK ~0 (GKs don't earn DC)
dc_base = {"DEF": 0.125, "MID": 0.136, "FWD": 0.058, "GK": 0.0}
need = asm["p_dc_hit"].isna()
asm.loc[need, "p_dc_hit"] = asm.loc[need, "position"].map(dc_base).fillna(0.10)

print("\nAfter fallback — all rows have p_dc_hit:", asm["p_dc_hit"].notna().all())
print("\nSample — top DC players (should be defensive specialists):")
print(asm.sort_values("p_dc_hit",ascending=False).drop_duplicates("element").head(6)[
    ["name","position","team","gw","p_dc_hit"]].to_string(index=False))

asm rows: 29452
Rows with a DC prediction: 9979 (33.9%)
Rows missing DC (need fallback): 19473

After fallback — all rows have p_dc_hit: True

Sample — top DC players (should be defensive specialists):
                               name position      team  gw  p_dc_hit
Joelinton Cássio Apolinário de Lira      MID Newcastle  10  0.798085
                        Xavi Simons      MID     Spurs  11  0.778342
   Rúben dos Santos Gato Alves Dias      DEF  Man City   9  0.770717
                 Dominik Szoboszlai      MID Liverpool   9  0.763716
                     Nathan Collins      DEF Brentford  22  0.751665
                       Josh Laurent      MID   Burnley  23  0.745286


In [19]:
# === FIX: dedupe DC join (asm grew 29,338 -> 29,452 = double-GW DC rows) ===
print("Before:", len(asm), "| duplicate (element,gw):", asm.duplicated(["element","gw"]).sum())

# DC per-match rows in a double-GW create duplicates. Collapse: take max p_dc_hit per player-GW
# (if a player has 2 matches in a GW, use their higher DC chance — or mean; max is fine for a per-match event)
asm = (asm.sort_values("p_dc_hit", ascending=False)
       .drop_duplicates(["element","gw"], keep="first")
       .sort_values(["element","gw"])
       .reset_index(drop=True))

print("After:", len(asm), "| duplicate (element,gw):", asm.duplicated(["element","gw"]).sum())
print("\nColumns now on asm:")
print([c for c in asm.columns if c in ["e_minutes","p_start","p60","npxg90","xa90","team_lambda","p_cs","p_dc_hit"]])

Before: 29452 | duplicate (element,gw): 114
After: 29338 | duplicate (element,gw): 0

Columns now on asm:
['p_start', 'p60', 'e_minutes', 'npxg90', 'xa90', 'team_lambda', 'p_cs', 'p_dc_hit']


In [20]:
# === ASSEMBLY CELL 8: The master equation → E[points] (without bonus yet) ===

# Scoring constants (2025-26)
GOAL_PTS = {"FWD":4, "MID":5, "DEF":6, "GK":6}
CS_PTS   = {"FWD":0, "MID":1, "DEF":4, "GK":4}

a = asm.copy()
a["minutes_frac"] = a["e_minutes"] / 90.0            # fraction of a full match expected

# Fixture scaling: scale attacking output by team's expected goals vs league average
LEAGUE_AVG_LAMBDA = 1.40
a["fixture_scale"] = a["team_lambda"] / LEAGUE_AVG_LAMBDA
a["fixture_scale"] = a["fixture_scale"].fillna(1.0).clip(0.5, 2.0)   # sane bounds

# --- Expected goals & assists this gameweek ---
a["e_goals"]   = a["npxg90"] * a["minutes_frac"] * a["fixture_scale"]
a["e_assists"] = a["xa90"]   * a["minutes_frac"] * a["fixture_scale"]

# --- Points components ---
# Appearance: 2 pts if plays 60+, else 1 pt if plays at all.
# p_start*p60 approximates P(60+); playing prob ~ p_start + (1-p_start)*p_sub (already in e_minutes>0)
a["playing_prob"] = 1 - (1 - a["p_start"])          # simplification: use p_start as "likely plays"
a["pts_appear"]   = a["p_start"] * a["p60"] * 2 + a["p_start"] * (1 - a["p60"]) * 1

a["pts_goals"]   = a["e_goals"]   * a["position"].map(GOAL_PTS)
a["pts_assists"] = a["e_assists"] * 3
a["pts_cs"]      = a["p_cs"] * a["position"].map(CS_PTS) * a["p60"]   # needs 60+ mins
a["pts_dc"]      = a["p_dc_hit"] * 2

# Sum (bonus added next cell)
a["e_points_nobonus"] = (a["pts_appear"] + a["pts_goals"] + a["pts_assists"]
                         + a["pts_cs"] + a["pts_dc"])

print("E[points] (no bonus yet) — summary:")
print(a["e_points_nobonus"].describe().round(2))
print("\nTop predicted players (no bonus):")
print(a.sort_values("e_points_nobonus",ascending=False).head(10)[
    ["name","position","team","gw","e_minutes","e_goals","pts_goals","pts_cs","e_points_nobonus"]].to_string(index=False))

E[points] (no bonus yet) — summary:
count    29338.00
mean         1.57
std          1.22
min          0.01
25%          0.55
50%          1.21
75%          2.37
max          8.45
Name: e_points_nobonus, dtype: float64

Top predicted players (no bonus):
          name position     team  gw  e_minutes  e_goals  pts_goals   pts_cs  e_points_nobonus
Erling Haaland      FWD Man City  23  84.684045 1.413583   5.654331 0.000000          8.449465
Erling Haaland      FWD Man City  15  86.041377 1.371060   5.484239 0.000000          8.321092
Erling Haaland      FWD Man City  17  81.438781 1.359411   5.437645 0.000000          8.134104
Erling Haaland      FWD Man City   6  81.417652 1.329167   5.316668 0.000000          8.000435
Erling Haaland      FWD Man City  13  85.218510 1.290369   5.161477 0.000000          7.912145
Erling Haaland      FWD Man City  21  83.035911 1.169960   4.679840 0.000000          7.310346
Erling Haaland      FWD Man City  36  77.728944 1.196008   4.784033 0.000000     

In [22]:
# %% [markdown]
# # Bonus (BPS) — Clean Prediction Pipeline
#
# Final config only. The investigation (algorithm bake-off, Core-Insights enrichment
# attempt/abandonment) is in `Bonus model log.md`. Deliberately simple per master
# plan §3.4 — bonus is a small contribution (max 3, ~11% of appearances earn any).
#
# **Two-piece design:**
# ```
# predicted components (goals, assists, CS, minutes + position, saves/cards/conceded)
#     │  Piece 1: LightGBM → predicted BPS
#     ▼  Piece 2: empirical BPS→bonus curve → expected bonus
# ```
#
# **Final config (verified against the log):**
# - **Piece 1:** LightGBM (300 trees, 31 leaves) on components + position dummies +
#   saves/cards/goals-conceded/penalties/own-goals. Trees beat linear (R² 0.747 vs
#   0.652) — real positional interactions. Trained on actual historical components;
#   at assembly, feed *predicted* components into the same model.
# - **Piece 2:** bucket BPS by 5, average actual bonus → expected-bonus lookup.
#   Flat ~0 below BPS 20, steep 20–40, plateau ~3 above 45.
# - **Data:** vaastav `all_seasons_fixed.parquet` (Core-Insights BPS abandoned —
#   Bug #8: corrupted values + grain mismatch).
#
# **Output:** `expected_bonus(components_frame)` → expected bonus per player-row.

# %%
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score

BASE = r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot"

# %% [markdown]
# ## 1. Load data

# %%
df = pd.read_parquet(BASE + r"\data\history\all_seasons_fixed.parquet")
d = df[(df["position"] != "AM") & (df["minutes"] >= 1)].copy()
for c in ["bps", "bonus", "minutes", "goals_scored", "assists", "clean_sheets",
          "saves", "yellow_cards", "red_cards", "goals_conceded", "penalties_missed", "own_goals"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")
print(f"Player-matches (appeared): {len(d)}")

# %% [markdown]
# ## 2. Piece 2 — the empirical BPS → expected-bonus curve
# For each BPS level (bucket of 5), the average bonus earned = expected bonus.
# Taking the average handles the "relative within a match" nature automatically.

# %%
d["bps_bin"] = (d["bps"] // 5) * 5
curve = (d.groupby("bps_bin")
         .agg(exp_bonus=("bonus", "mean"), n=("bonus", "size"))
         .reset_index())
curve = curve[curve["n"] >= 30]

# lookup: map a BPS value to expected bonus (clip to the curve's range)
_bps_bins = curve["bps_bin"].values
_exp_bonus = curve["exp_bonus"].values

def bps_to_bonus(bps_values):
    """Map predicted BPS -> expected bonus via the empirical curve (interpolated)."""
    b = np.clip(np.asarray(bps_values, dtype=float), _bps_bins.min(), _bps_bins.max())
    return np.interp(b, _bps_bins, _exp_bonus)

print("BPS -> expected bonus curve (key points):")
for bps in [10, 20, 25, 30, 35, 45, 60]:
    print(f"  BPS {bps:3d} -> E[bonus] {bps_to_bonus(bps):.3f}")

# %% [markdown]
# ## 3. Piece 1 — predict BPS from components (LightGBM)
# Trained on ACTUAL components; the BPS→component relationship is a fixed property
# of the scoring system, so it applies identically to predicted inputs at assembly.

# %%
comp = ["goals_scored", "assists", "clean_sheets", "minutes"]
extra = ["saves", "yellow_cards", "red_cards", "goals_conceded", "penalties_missed", "own_goals"]
d["is_def"] = (d["position"] == "DEF").astype(int)
d["is_mid"] = (d["position"] == "MID").astype(int)
d["is_gk"]  = (d["position"] == "GK").astype(int)
BPS_FEATURES = comp + ["is_def", "is_mid", "is_gk"] + extra

mdf = d.dropna(subset=BPS_FEATURES + ["bps"]).copy()
tr = mdf[mdf["season"] <= "2023-24"]
te = mdf[mdf["season"] == "2024-25"]

bps_model = lgb.LGBMRegressor(n_estimators=300, num_leaves=31, learning_rate=0.05,
                              random_state=42, verbose=-1)
bps_model.fit(tr[BPS_FEATURES], tr["bps"])
pred = bps_model.predict(te[BPS_FEATURES])
print(f"BPS model — MAE {mean_absolute_error(te['bps'], pred):.2f}  R² {r2_score(te['bps'], pred):.3f}  (log: MAE 4.19, R² 0.747)")

# %% [markdown]
# ## 4. The full chain + the reusable function

# %%
def expected_bonus(frame):
    """Given a frame with the BPS component columns, return expected bonus per row.
    At assembly, `frame` carries PREDICTED components (E[goals], E[assists], etc.);
    here we validate on actuals. Missing extra columns are filled 0."""
    f = frame.copy()
    for col in BPS_FEATURES:
        if col not in f.columns:
            f[col] = 0
    f[BPS_FEATURES] = f[BPS_FEATURES].fillna(0)
    f["pred_bps"] = bps_model.predict(f[BPS_FEATURES])
    f["exp_bonus"] = bps_to_bonus(f["pred_bps"].values)
    return f

# Validate the full chain on 2024-25: predicted expected-bonus vs actual bonus
val = expected_bonus(te)
print(f"\nFull chain on 2024-25 ({len(val)} rows):")
print(f"  mean predicted expected-bonus: {val['exp_bonus'].mean():.3f}")
print(f"  mean actual bonus:             {te['bonus'].mean():.3f}")
print(f"  correlation (pred exp-bonus vs actual bonus): {val['exp_bonus'].corr(te['bonus']):.3f}")

print("\nHighest expected-bonus players (2024-25 sample):")
show = val.sort_values("exp_bonus", ascending=False)[["position","goals_scored","assists","clean_sheets","pred_bps","exp_bonus","bonus"]].head(6)
print(show.to_string(index=False))

Player-matches (appeared): 108687
BPS -> expected bonus curve (key points):
  BPS  10 -> E[bonus] 0.000
  BPS  20 -> E[bonus] 0.160
  BPS  25 -> E[bonus] 0.752
  BPS  30 -> E[bonus] 1.692
  BPS  35 -> E[bonus] 2.246
  BPS  45 -> E[bonus] 2.668
  BPS  60 -> E[bonus] 2.898
BPS model — MAE 4.19  R² 0.747  (log: MAE 4.19, R² 0.747)

Full chain on 2024-25 (11566 rows):
  mean predicted expected-bonus: 0.227
  mean actual bonus:             0.208
  correlation (pred exp-bonus vs actual bonus): 0.763

Highest expected-bonus players (2024-25 sample):
position  goals_scored  assists  clean_sheets  pred_bps  exp_bonus  bonus
     FWD             3        0             1 80.003711   2.970588      3
     FWD             3        0             0 83.892677   2.970588      3
     FWD             3        0             1 79.521531   2.970588      3
     MID             3        1             0 77.248643   2.970588      3
     FWD             3        0             1 79.874372   2.970588      3
     MI

In [23]:
# === ASSEMBLY CELL 9: Bonus + final E[points] ===
# Feed predicted components into the BPS model -> predicted BPS -> expected-bonus curve.

# The BPS model (from bonus_predict.py) expects these columns:
# goals_scored, assists, clean_sheets, minutes, is_def/is_mid/is_gk, saves/cards/conceded...
# We supply the EXPECTED versions (predicted), zeros for ones we don't model (cards, saves).
bps_input = pd.DataFrame({
    "goals_scored":  a["e_goals"],
    "assists":       a["e_assists"],
    "clean_sheets":  a["p_cs"] * a["p60"],          # expected clean sheets
    "minutes":       a["e_minutes"],
    "is_def": (a["position"]=="DEF").astype(int),
    "is_mid": (a["position"]=="MID").astype(int),
    "is_gk":  (a["position"]=="GK").astype(int),
    "saves": 0, "yellow_cards": 0, "red_cards": 0,
    "goals_conceded": 0, "penalties_missed": 0, "own_goals": 0,
})

# Predict BPS, map to expected bonus via the curve (both from the bonus model)
a["pred_bps"]   = bps_model.predict(bps_input[BPS_FEATURES])
a["exp_bonus"]  = bps_to_bonus(a["pred_bps"].values)

# FINAL expected points
a["e_points"] = a["e_points_nobonus"] + a["exp_bonus"]

print("FINAL E[points] — summary:")
print(a["e_points"].describe().round(2))
print("\n=== TOP PREDICTED PLAYERS (full E[points] with bonus) ===")
print(a.sort_values("e_points",ascending=False).head(12)[
    ["name","position","team","gw","e_minutes","e_points_nobonus","exp_bonus","e_points"]].to_string(index=False))

FINAL E[points] — summary:
count    29338.00
mean         4.01
std          1.33
min          2.37
25%          2.84
50%          3.66
75%          4.86
max         10.91
Name: e_points, dtype: float64

=== TOP PREDICTED PLAYERS (full E[points] with bonus) ===
                        name position     team  gw  e_minutes  e_points_nobonus  exp_bonus  e_points
              Erling Haaland      FWD Man City  23  84.684045          8.449465   2.458618 10.908084
              Erling Haaland      FWD Man City  15  86.041377          8.321092   2.480307 10.801399
              Erling Haaland      FWD Man City  17  81.438781          8.134104   2.506856 10.640960
              Erling Haaland      FWD Man City   6  81.417652          8.000435   2.506856 10.507291
              Erling Haaland      FWD Man City  13  85.218510          7.912145   2.458618 10.370763
              Erling Haaland      FWD Man City  36  77.728944          7.290144   2.500343  9.790487
              Erling Haaland    

In [24]:
# === FIX: bonus must scale with playing probability (bench players shouldn't get ~2.4) ===
# The BPS model predicts BPS assuming the player plays those minutes. But e_minutes already
# encodes low minutes for bench players — the issue is the curve's floor. Scale bonus by
# the fraction of a full match expected, so near-zero-minute players get near-zero bonus.

a["exp_bonus_raw"] = a["exp_bonus"]                          # keep the unscaled version
a["exp_bonus"] = a["exp_bonus"] * (a["e_minutes"] / 90.0).clip(0, 1)   # scale by playing time

a["e_points"] = a["e_points_nobonus"] + a["exp_bonus"]

print("Corrected E[points]:")
print(a["e_points"].describe().round(2))
print("\nCheck — a bench player (low e_minutes) should now have low bonus:")
print(a[a["e_minutes"] < 10].head(3)[["name","e_minutes","exp_bonus_raw","exp_bonus","e_points"]].to_string(index=False))
print("\nTop players (should be ~unchanged):")
print(a.sort_values("e_points",ascending=False).head(6)[
    ["name","position","team","gw","e_minutes","e_points_nobonus","exp_bonus","e_points"]].to_string(index=False))

Corrected E[points]:
count    29338.00
mean         2.28
std          2.04
min          0.02
25%          0.60
50%          1.40
75%          3.83
max         10.76
Name: e_points, dtype: float64

Check — a bench player (low e_minutes) should now have low bonus:
                      name  e_minutes  exp_bonus_raw  exp_bonus  e_points
Kepa Arrizabalaga Revuelta   4.858022       2.412125   0.130202  1.843168
Kepa Arrizabalaga Revuelta   3.089176       2.409155   0.082692  1.050082
Kepa Arrizabalaga Revuelta   1.857169       2.407272   0.049675  1.759007

Top players (should be ~unchanged):
          name position     team  gw  e_minutes  e_points_nobonus  exp_bonus  e_points
Erling Haaland      FWD Man City  23  84.684045          8.449465   2.313397 10.762862
Erling Haaland      FWD Man City  15  86.041377          8.321092   2.371211 10.692303
Erling Haaland      FWD Man City  17  81.438781          8.134104   2.268392 10.402496
Erling Haaland      FWD Man City   6  81.417652         

In [25]:
print(a.sort_values("e_points",ascending=False).drop_duplicates("element").head(15)[
    ["name","position","team","e_points"]].to_string(index=False))

                        name position      team  e_points
              Erling Haaland      FWD  Man City 10.762862
               Nico O'Reilly      DEF  Man City  9.171420
Gabriel dos Santos Magalhães      DEF   Arsenal  9.128550
                 Bukayo Saka      MID   Arsenal  9.085885
                Malick Thiaw      DEF Newcastle  8.935929
                Bryan Mbeumo      MID   Man Utd  8.890303
           Tijjani Reijnders      MID  Man City  8.636465
               Mohamed Salah      MID Liverpool  8.626676
      Bruno Borges Fernandes      MID   Man Utd  8.498934
             Antoine Semenyo      MID  Man City  8.362514
              Nathan Collins      DEF Brentford  8.345291
             Virgil van Dijk      DEF Liverpool  8.337502
              Jurriën Timber      DEF   Arsenal  8.329195
               Patrick Dorgu      DEF   Man Utd  8.320955
       Marc Cucurella Saseta      DEF   Chelsea  8.313345


In [26]:
# === ASSEMBLY CELL 10: VALIDATION — does predicted E[points] match actual? ===
# We have actual_points on the skeleton (from vaastav 2025-26). Compare.

val = a[a["actual_points"].notna()].copy()
val["actual_points"] = pd.to_numeric(val["actual_points"], errors="coerce")
val = val.dropna(subset=["actual_points","e_points"])
print(f"Player-gameweeks with actuals: {len(val)}")

# 1. Overall correlation & error
from scipy.stats import spearmanr
pearson = val["e_points"].corr(val["actual_points"])
spearman = spearmanr(val["e_points"], val["actual_points"]).correlation
mae = (val["e_points"] - val["actual_points"]).abs().mean()
print(f"\nPredicted vs Actual:")
print(f"  Pearson correlation:  {pearson:.3f}")
print(f"  Spearman (rank) corr: {spearman:.3f}")
print(f"  MAE: {mae:.2f} points")
print(f"  Mean predicted: {val['e_points'].mean():.2f} | Mean actual: {val['actual_points'].mean():.2f}")

# 2. Calibration by predicted bucket — do high-predicted players actually score more?
val["pred_bucket"] = pd.cut(val["e_points"], [0,1,2,3,4,5,20],
                            labels=["0-1","1-2","2-3","3-4","4-5","5+"])
print("\nCalibration — mean actual points by predicted bucket:")
print(val.groupby("pred_bucket", observed=True)["actual_points"].agg(["mean","count"]).round(2))

# 3. Does it beat a naive baseline (predict everyone's season-avg)?
naive = val.groupby("element")["actual_points"].transform("mean")
naive_mae = (naive - val["actual_points"]).abs().mean()
print(f"\nMAE — our model: {mae:.2f}  vs  naive season-average: {naive_mae:.2f}")

Player-gameweeks with actuals: 29338

Predicted vs Actual:
  Pearson correlation:  0.565
  Spearman (rank) corr: 0.667
  MAE: 1.69 points
  Mean predicted: 2.28 | Mean actual: 1.17

Calibration — mean actual points by predicted bucket:
             mean  count
pred_bucket             
0-1          0.12  11216
1-2          0.35   6756
2-3          1.39   2284
3-4          2.15   2111
4-5          2.75   2408
5+           3.60   4563

MAE — our model: 1.69  vs  naive season-average: 1.01


In [27]:
# === DIAGNOSE: where is the over-prediction concentrated? ===
val["resid"] = val["e_points"] - val["actual_points"]

# By playing likelihood
val["mins_band"] = pd.cut(val["e_minutes"], [0,15,45,70,90], labels=["<15","15-45","45-70","70+"])
print("Over-prediction by expected-minutes band:")
print(val.groupby("mins_band", observed=True).agg(
    pred=("e_points","mean"), actual=("actual_points","mean"),
    over=("resid","mean"), n=("resid","size")).round(2))

# The key question: for NAILED players (70+ min), is it well-calibrated?
nailed = val[val["e_minutes"]>=70]
print(f"\nNailed players (70+ e_min): pred {nailed['e_points'].mean():.2f}  actual {nailed['actual_points'].mean():.2f}")

Over-prediction by expected-minutes band:
           pred  actual  over      n
mins_band                           
<15        0.83    0.13  0.70  16770
15-45      2.33    1.44  0.88   4099
45-70      4.30    2.63  1.67   4039
70+        5.90    3.54  2.36   4430

Nailed players (70+ e_min): pred 5.90  actual 3.54


In [28]:
# === DIAGNOSE: which scoring COMPONENT is over-predicting? ===
# Compare each predicted component's mean to what actually happened, for nailed players.
nailed = val[val["e_minutes"]>=70].copy()

# actual breakdowns from vaastav (we have these columns in df for 2025-26)
actual_cols = df[df["season"]=="2025-26"][["element","GW","goals_scored","assists",
              "clean_sheets","bonus","total_points"]].copy()
actual_cols = actual_cols.rename(columns={"GW":"gw"})
actual_cols = actual_cols.groupby(["element","gw"]).sum().reset_index()  # collapse doubles
nd = nailed.merge(actual_cols, on=["element","gw"], how="left")

print("Nailed players — PREDICTED vs ACTUAL by component (mean per player-GW):\n")
print(f"{'component':14s} {'predicted':>10s} {'actual':>8s}")
print(f"{'goals pts':14s} {nd['pts_goals'].mean():>10.2f} {(nd['goals_scored'].fillna(0)*nd['position'].map(GOAL_PTS)).mean():>8.2f}")
print(f"{'assist pts':14s} {nd['pts_assists'].mean():>10.2f} {(nd['assists'].fillna(0)*3).mean():>8.2f}")
print(f"{'CS pts':14s} {nd['pts_cs'].mean():>10.2f} {(nd['clean_sheets'].fillna(0)*nd['position'].map(CS_PTS)).mean():>8.2f}")
print(f"{'DC pts':14s} {nd['pts_dc'].mean():>10.2f} {'?':>8s}  (no clean actual col)")
print(f"{'appearance':14s} {nd['pts_appear'].mean():>10.2f} {'~1.9':>8s}  (most play 60+)")
print(f"{'bonus':14s} {nd['exp_bonus'].mean():>10.2f} {nd['bonus'].fillna(0).mean():>8.2f}")

Nailed players — PREDICTED vs ACTUAL by component (mean per player-GW):

component       predicted   actual
goals pts            0.53     0.44
assist pts           0.25     0.27
CS pts               0.74     0.65
DC pts               0.32        ?  (no clean actual col)
appearance           1.77     ~1.9  (most play 60+)
bonus                2.28     0.27


In [29]:
# === FIX: bonus is ~8x over-predicted. Rescale to match actual bonus rate. ===
# Bonus should be small. Everything else is well-calibrated; only bonus is broken.
overall_pred_bonus = a["exp_bonus"].mean()
# actual bonus mean across all appearances (from the data)
actual_bonus_mean = pd.to_numeric(df[df["season"]=="2025-26"]["bonus"], errors="coerce").mean()
scale = actual_bonus_mean / overall_pred_bonus
print(f"Predicted bonus mean: {overall_pred_bonus:.3f}")
print(f"Actual bonus mean:    {actual_bonus_mean:.3f}")
print(f"Rescale factor:       {scale:.3f}")

a["exp_bonus"] = a["exp_bonus"] * scale
a["e_points"] = a["e_points_nobonus"] + a["exp_bonus"]

# Re-validate
val = a[a["actual_points"].notna()].copy()
val["actual_points"] = pd.to_numeric(val["actual_points"], errors="coerce")
val = val.dropna(subset=["actual_points","e_points"])
from scipy.stats import spearmanr
print(f"\nAfter bonus fix:")
print(f"  Mean predicted: {val['e_points'].mean():.2f} | Mean actual: {val['actual_points'].mean():.2f}")
print(f"  Pearson: {val['e_points'].corr(val['actual_points']):.3f}")
print(f"  Spearman: {spearmanr(val['e_points'], val['actual_points']).correlation:.3f}")
print(f"  MAE: {(val['e_points']-val['actual_points']).abs().mean():.2f}")

Predicted bonus mean: 0.712
Actual bonus mean:    0.081
Rescale factor:       0.114

After bonus fix:
  Mean predicted: 1.65 | Mean actual: 1.17
  Pearson: 0.550
  Spearman: 0.629
  MAE: 1.38


In [30]:
# Re-check the nailed-player calibration after the bonus fix
val["mins_band"] = pd.cut(val["e_minutes"], [0,15,45,70,90], labels=["<15","15-45","45-70","70+"])
print("After bonus fix — predicted vs actual by minutes band:")
print(val.groupby("mins_band", observed=True).agg(
    pred=("e_points","mean"), actual=("actual_points","mean"),
    over=("e_points","mean")).round(2))
print("\nvs naive baseline — but note the naive uses FULL-SEASON avg (has future info, unfair):")
print("A fairer baseline: predict each player's average from PRIOR gameweeks only")

# fair baseline: expanding mean (only past games)
val_sorted = val.sort_values(["element","gw"])
val_sorted["fair_naive"] = val_sorted.groupby("element")["actual_points"].transform(
    lambda s: s.shift(1).expanding().mean())
fair = val_sorted.dropna(subset=["fair_naive"])
print(f"\nMAE our model:      {(fair['e_points']-fair['actual_points']).abs().mean():.2f}")
print(f"MAE fair baseline:  {(fair['fair_naive']-fair['actual_points']).abs().mean():.2f}  (prior-games avg, no leakage)")

After bonus fix — predicted vs actual by minutes band:
           pred  actual  over
mins_band                    
<15        0.78    0.13  0.78
15-45      1.63    1.44  1.63
45-70      2.87    2.63  2.87
70+        3.87    3.54  3.87

vs naive baseline — but note the naive uses FULL-SEASON avg (has future info, unfair):
A fairer baseline: predict each player's average from PRIOR gameweeks only

MAE our model:      1.37
MAE fair baseline:  1.07  (prior-games avg, no leakage)


In [32]:
# === REAL FIX: gate every performance term by playing probability ===
# Bug: DC and appearance gave points to players who won't play. Fix: gate by minutes.
a = asm.copy()

a["minutes_frac"] = (a["e_minutes"] / 90.0).clip(0, 1)
LEAGUE_AVG_LAMBDA = 1.40
a["fixture_scale"] = (a["team_lambda"] / LEAGUE_AVG_LAMBDA).fillna(1.0).clip(0.5, 2.0)

GOAL_PTS = {"FWD":4,"MID":5,"DEF":6,"GK":6}
CS_PTS   = {"FWD":0,"MID":1,"DEF":4,"GK":4}

# goals/assists — scale with minutes ✓
a["e_goals"]   = a["npxg90"] * a["minutes_frac"] * a["fixture_scale"]
a["e_assists"] = a["xa90"]   * a["minutes_frac"] * a["fixture_scale"]
a["pts_goals"]   = a["e_goals"]   * a["position"].map(GOAL_PTS)
a["pts_assists"] = a["e_assists"] * 3

# appearance — 3-state, properly gated
a["p_60plus"]   = a["p_start"] * a["p60"]
a["p_play_any"] = a["p_start"] + (1 - a["p_start"]) * 0.30    # start OR come off bench
a["pts_appear"] = a["p_60plus"] * 2 + (a["p_play_any"] - a["p_60plus"]).clip(lower=0) * 1

# clean sheet — needs 60+, gate by p_60plus
a["pts_cs"] = a["p_cs"] * a["position"].map(CS_PTS) * a["p_60plus"]

# DC — gate by minutes (no minutes = no DC)
a["pts_dc"] = a["p_dc_hit"] * 2 * a["minutes_frac"]

a["e_points_core"] = a["pts_appear"] + a["pts_goals"] + a["pts_assists"] + a["pts_cs"] + a["pts_dc"]

# --- check the fringe band using actuals straight from asm ---
a["actual_points"] = pd.to_numeric(a["actual_points"], errors="coerce")
a["mins_band"] = pd.cut(a["e_minutes"], [0,15,45,70,90], labels=["<15","15-45","45-70","70+"])
print("Core points (no bonus) by minutes band — is the <15 fringe fixed?")
print(a.groupby("mins_band", observed=True).agg(
    pred=("e_points_core","mean"), actual=("actual_points","mean"), n=("e_points_core","size")).round(2))

Core points (no bonus) by minutes band — is the <15 fringe fixed?
           pred  actual      n
mins_band                     
<15        0.37    0.13  16770
15-45      1.32    1.44   4099
45-70      2.56    2.63   4039
70+        3.54    3.54   4430


In [33]:
# === Re-add bonus (gated by minutes) + final validation ===
# Bonus: predicted BPS -> curve -> scale by minutes_frac (must be on pitch to earn bonus)
bps_input = pd.DataFrame({
    "goals_scored": a["e_goals"], "assists": a["e_assists"],
    "clean_sheets": a["p_cs"]*a["p_60plus"], "minutes": a["e_minutes"],
    "is_def":(a["position"]=="DEF").astype(int), "is_mid":(a["position"]=="MID").astype(int),
    "is_gk":(a["position"]=="GK").astype(int),
    "saves":0,"yellow_cards":0,"red_cards":0,"goals_conceded":0,"penalties_missed":0,"own_goals":0})
a["pred_bps"] = bps_model.predict(bps_input[BPS_FEATURES])
a["exp_bonus"] = bps_to_bonus(a["pred_bps"].values) * a["minutes_frac"]

# recalibrate bonus to actual rate (bonus is small; keep it honest)
actual_bonus_mean = pd.to_numeric(df[df["season"]=="2025-26"]["bonus"], errors="coerce").mean()
a["exp_bonus"] *= actual_bonus_mean / a["exp_bonus"].mean()

a["e_points"] = a["e_points_core"] + a["exp_bonus"]

# === FULL VALIDATION ===
from scipy.stats import spearmanr
v = a.dropna(subset=["actual_points","e_points"]).copy()
print("=== FINAL VALIDATION ===")
print(f"Pearson:  {v['e_points'].corr(v['actual_points']):.3f}")
print(f"Spearman: {spearmanr(v['e_points'], v['actual_points']).correlation:.3f}")
print(f"MAE:      {(v['e_points']-v['actual_points']).abs().mean():.2f}")
print(f"Mean pred {v['e_points'].mean():.2f} | actual {v['actual_points'].mean():.2f}")

# fair baseline (prior-games only, no leakage)
vs = v.sort_values(["element","gw"])
vs["fair_naive"] = vs.groupby("element")["actual_points"].transform(lambda s: s.shift(1).expanding().mean())
f = vs.dropna(subset=["fair_naive"])
print(f"\nMAE our model: {(f['e_points']-f['actual_points']).abs().mean():.2f}  vs fair baseline: {(f['fair_naive']-f['actual_points']).abs().mean():.2f}")

=== FINAL VALIDATION ===
Pearson:  0.581
Spearman: 0.716
MAE:      1.13
Mean pred 1.36 | actual 1.17

MAE our model: 1.12  vs fair baseline: 1.07
